# Bronze Layer

In [0]:
df = spark.table("workspace.default.data_co_supply_chain_dataset")
print(df.count(), len(df.columns))

In [0]:
df.printSchema()

In [0]:
display(df.limit(5))

In [0]:
import re

# Clean column names: lowercase, replace spaces with _, remove special characters
clean_columns = [re.sub(r'[#/(),;{}]', '', col).strip().replace(' ', '_').lower() 
                 for col in df.columns]

# Apply new column names to the DataFrame
bronze_df = df.toDF(*clean_columns)

# Verify the new column names
bronze_df.printSchema()

In [0]:
bronze_df = (
    bronze_df
    .withColumn("order_date", F.to_timestamp(F.col("order_date_dateorders"), "M/d/yyyy H:mm"))
    .withColumn("shipping_date", F.to_timestamp(F.col("shipping_date_dateorders"), "M/d/yyyy H:mm"))
    .drop(
        "order_date_dateorders",
        "shipping_date_dateorders",
        "customer_email",
        "customer_password",
        "product_image",
        "product_description"
    )
)

In [0]:
bronze_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_dataco")

print(f"Bronze layer saved: {bronze_df.count()} rows x {len(bronze_df.columns)} columns")